# UC-02: Invoice Three-Way Match — Exploratory Data Analysis

**Objective:** Understand the distribution, patterns, and relationships in the invoice matching data
to inform feature engineering and model selection.

**Key questions:**
- How is match_status distributed? Is binary (FULL_MATCH vs ANY_VARIANCE) viable?
- Which vendor/PO/GR characteristics correlate with match failures?
- Are there temporal patterns in match rates?
- What features risk target leakage?

## 1. Setup & Data Loading

In [ ]:
import sys
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parents[2]
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid", palette="muted")
pd.set_option("display.max_columns", 50)

from ml.common.db_config import load_tables
from ml.data_processing.python.uc02_preprocessing import (
    UC02_TABLES, load_uc02_raw_data, build_uc02_base_dataset,
    add_temporal_features,
)

In [ ]:
# Load all tables
csv_dir = project_root / "output" / "csv"
tables = load_uc02_raw_data(source="csv", csv_dir=csv_dir)

print("Table shapes:")
for name, df in tables.items():
    print(f"  {name:30s} {df.shape}")

In [ ]:
# Build base dataset
df = build_uc02_base_dataset(tables)
df = add_temporal_features(df)
print(f"Base dataset: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

## 2. Target Variable Distribution

In [ ]:
# 4-class distribution
match_counts = df["match_status"].value_counts()
match_pct = df["match_status"].value_counts(normalize=True) * 100

print("Match Status Distribution:")
for status in ["FULL_MATCH", "PRICE_VARIANCE", "QUANTITY_VARIANCE", "BOTH_VARIANCE"]:
    count = match_counts.get(status, 0)
    pct = match_pct.get(status, 0)
    print(f"  {status:20s}: {count:4d} ({pct:.1f}%)")

# Binary distribution
binary_target = (df["match_status"] != "FULL_MATCH").astype(int)
print(f"\nBinary: FULL_MATCH={int((binary_target == 0).sum())}, ANY_VARIANCE={int(binary_target.sum())}")
print(f"Imbalance ratio: {(binary_target == 0).sum() / max(binary_target.sum(), 1):.1f}:1")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart — 4-class
colors = ["#2ecc71", "#e74c3c", "#f39c12", "#9b59b6"]
order = ["FULL_MATCH", "PRICE_VARIANCE", "QUANTITY_VARIANCE", "BOTH_VARIANCE"]
sns.countplot(data=df, x="match_status", order=order, palette=colors, ax=axes[0])
axes[0].set_title("Invoice Match Status (4-class)")
axes[0].set_xlabel("")
for p in axes[0].patches:
    axes[0].annotate(f"{int(p.get_height())}", (p.get_x() + p.get_width()/2, p.get_height()),
                     ha="center", va="bottom")

# Pie chart — binary
binary_counts = [int((binary_target == 0).sum()), int(binary_target.sum())]
axes[1].pie(binary_counts, labels=["FULL_MATCH", "ANY_VARIANCE"],
            autopct="%1.1f%%", colors=["#2ecc71", "#e74c3c"], startangle=90)
axes[1].set_title("Binary Target Distribution")

plt.tight_layout()
plt.show()

## 3. Temporal Distribution

In [ ]:
# Invoice count by month
df["invoice_date_dt"] = pd.to_datetime(df["invoice_date"])
df["year_month"] = df["invoice_date_dt"].dt.to_period("M").astype(str)

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Monthly volume
monthly = df.groupby("year_month").size()
monthly.plot(kind="bar", ax=axes[0], color="steelblue")
axes[0].set_title("Invoice Volume by Month")
axes[0].set_ylabel("Count")
axes[0].tick_params(axis="x", rotation=45)

# Stacked bar by quarter — match status breakdown
quarter_status = pd.crosstab(df["invoice_quarter"], df["match_status"])
quarter_status = quarter_status.reindex(columns=order, fill_value=0)
quarter_status.plot(kind="bar", stacked=True, ax=axes[1], color=colors)
axes[1].set_title("Match Status by Quarter")
axes[1].set_ylabel("Count")
axes[1].legend(title="Match Status", bbox_to_anchor=(1.02, 1))

plt.tight_layout()
plt.show()

## 4. Vendor Quality vs Match Rate

In [ ]:
# Merge vendor scores
vendor_master = tables["vendor_master"]
vendor_scores = vendor_master[["vendor_id", "quality_score", "risk_score",
                                "on_time_delivery_rate", "esg_score"]].copy()
df_v = df.merge(vendor_scores, on="vendor_id", how="left")

# Per-vendor match rate (min 3 invoices)
vendor_agg = df_v.groupby("vendor_id").agg(
    invoice_count=("invoice_id", "count"),
    match_rate=("match_status", lambda x: (x == "FULL_MATCH").mean()),
    quality_score=("quality_score", "first"),
    risk_score=("risk_score", "first"),
).reset_index()

vendor_agg_filtered = vendor_agg[vendor_agg["invoice_count"] >= 3]
print(f"Vendors with 3+ invoices: {len(vendor_agg_filtered)}")

# Scatter plot
fig = px.scatter(
    vendor_agg_filtered,
    x="quality_score", y="match_rate",
    size="invoice_count", color="risk_score",
    hover_data=["vendor_id"],
    title="Vendor Quality Score vs Invoice Match Rate (vendors with 3+ invoices)",
    labels={"quality_score": "Quality Score", "match_rate": "Full Match Rate",
            "risk_score": "Risk Score"},
    color_continuous_scale="RdYlGn_r",
)
fig.show()

In [ ]:
# Box plot: quality score by match status
fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data=df_v, x="match_status", y="quality_score", order=order, palette=colors, ax=ax)
ax.set_title("Vendor Quality Score by Match Status")
ax.set_xlabel("")
plt.tight_layout()
plt.show()

## 5. Risk Score Analysis

In [ ]:
# Risk vs Quality correlation
corr, pval = stats.pearsonr(
    vendor_master["quality_score"].dropna(),
    vendor_master["risk_score"].dropna(),
)
print(f"Quality-Risk correlation: r={corr:.3f}, p={pval:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter: risk vs quality
sns.scatterplot(data=vendor_master, x="quality_score", y="risk_score",
                alpha=0.6, ax=axes[0])
axes[0].set_title(f"Risk vs Quality Score (r={corr:.3f})")

# Box: risk score by match status
sns.boxplot(data=df_v, x="match_status", y="risk_score", order=order, palette=colors, ax=axes[1])
axes[1].set_title("Risk Score by Match Status")
axes[1].set_xlabel("")

plt.tight_layout()
plt.show()

## 6. Contract Coverage Impact

In [ ]:
# Contract coverage vs match rate
df["has_contract"] = df["contract_id"].notna().map({True: "On-Contract", False: "Off-Contract"})

contract_match = pd.crosstab(df["has_contract"], df["match_status"], normalize="index") * 100
print("Match rate by contract coverage:")
print(contract_match.round(1))

# Chi-square test
ct = pd.crosstab(df["has_contract"], df["match_status"])
chi2, p, dof, expected = stats.chi2_contingency(ct)
print(f"\nChi-square: {chi2:.2f}, p={p:.4f}, dof={dof}")

fig, ax = plt.subplots(figsize=(10, 5))
contract_match.reindex(columns=order).plot(kind="bar", ax=ax, color=colors)
ax.set_title("Match Status Distribution: On-Contract vs Off-Contract")
ax.set_ylabel("Percentage")
ax.legend(title="Match Status", bbox_to_anchor=(1.02, 1))
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()

## 7. PO Type and Incoterms

In [ ]:
# PO type cross-tab
print("Match status by PO type:")
print(pd.crosstab(df["po_type"], df["match_status"], margins=True))
print()

# Maverick flag cross-tab
print("Match status by maverick flag:")
print(pd.crosstab(df["maverick_flag"], df["match_status"], margins=True))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# PO type
po_type_match = pd.crosstab(df["po_type"], df["match_status"], normalize="index") * 100
po_type_match.reindex(columns=order).plot(kind="bar", stacked=True, ax=axes[0], color=colors)
axes[0].set_title("Match Status by PO Type")
axes[0].set_ylabel("Percentage")
axes[0].tick_params(axis="x", rotation=0)

# Maverick
mav_match = pd.crosstab(df["maverick_flag"], df["match_status"], normalize="index") * 100
mav_match.reindex(columns=order).plot(kind="bar", stacked=True, ax=axes[1], color=colors)
axes[1].set_title("Match Status by Maverick Flag")
axes[1].set_ylabel("Percentage")
axes[1].tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.show()

## 8. GR Quality Indicators

In [ ]:
# GR quantity / PO quantity ratio
if "quantity_received" in df.columns and "quantity" in df.columns:
    df["gr_po_ratio"] = pd.to_numeric(df["quantity_received"], errors="coerce") / \
                         pd.to_numeric(df["quantity"], errors="coerce")

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Distribution by match status
    for status, color in zip(order, colors):
        subset = df[df["match_status"] == status]["gr_po_ratio"].dropna()
        if len(subset) > 0:
            axes[0].hist(subset, bins=30, alpha=0.5, label=status, color=color)
    axes[0].set_title("GR/PO Quantity Ratio Distribution")
    axes[0].set_xlabel("Ratio")
    axes[0].legend()

    # Rejection rate by match status
    if "quantity_rejected" in df.columns:
        df["has_rejection"] = pd.to_numeric(df["quantity_rejected"], errors="coerce").fillna(0) > 0
        rej_rate = df.groupby("match_status")["has_rejection"].mean() * 100
        rej_rate.reindex(order).plot(kind="bar", ax=axes[1], color="coral")
        axes[1].set_title("% Invoices with GR Rejections")
        axes[1].set_ylabel("Percentage")
        axes[1].tick_params(axis="x", rotation=45)

    plt.tight_layout()
    plt.show()

## 9. Price and Amount Features

In [ ]:
# Unit price vs standard cost ratio
if "unit_price" in df.columns and "standard_cost" in df.columns:
    unit_price = pd.to_numeric(df["unit_price"], errors="coerce")
    std_cost = pd.to_numeric(df["standard_cost"], errors="coerce")
    df["price_std_ratio"] = np.where(std_cost > 0, unit_price / std_cost, np.nan)

    fig, ax = plt.subplots(figsize=(10, 5))
    for status, color in zip(order, colors):
        subset = df[df["match_status"] == status]["price_std_ratio"].dropna()
        if len(subset) > 0:
            ax.hist(subset, bins=30, alpha=0.5, label=status, color=color)
    ax.set_title("PO Unit Price / Standard Cost Ratio by Match Status")
    ax.set_xlabel("Price / Standard Cost")
    ax.legend()
    plt.tight_layout()
    plt.show()

## 10. Temporal Features

In [ ]:
# Days GR to invoice by match status
if "days_gr_to_invoice" in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    sns.boxplot(data=df, x="match_status", y="days_gr_to_invoice",
                order=order, palette=colors, ax=axes[0])
    axes[0].set_title("Days GR → Invoice by Match Status")
    axes[0].set_xlabel("")

    # Days PO to invoice
    if "days_po_to_invoice" in df.columns:
        sns.boxplot(data=df, x="match_status", y="days_po_to_invoice",
                    order=order, palette=colors, ax=axes[1])
        axes[1].set_title("Days PO → Invoice by Match Status")
        axes[1].set_xlabel("")

    plt.tight_layout()
    plt.show()

## 11. Correlation Matrix

In [ ]:
# Select numeric columns for correlation
numeric_cols = df_v.select_dtypes(include=[np.number]).columns.tolist()
# Exclude ID-like columns
exclude_patterns = ["_number", "_line"]
numeric_cols = [c for c in numeric_cols if not any(p in c for p in exclude_patterns)]

if len(numeric_cols) > 2:
    corr_matrix = df_v[numeric_cols].corr()

    # Seaborn heatmap
    fig, ax = plt.subplots(figsize=(16, 12))
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
    sns.heatmap(corr_matrix, mask=mask, annot=False, cmap="RdBu_r",
                center=0, vmin=-1, vmax=1, ax=ax)
    ax.set_title("Feature Correlation Matrix")
    plt.tight_layout()
    plt.show()

    # Plotly interactive
    fig = px.imshow(corr_matrix, color_continuous_scale="RdBu_r",
                    zmin=-1, zmax=1, title="Feature Correlations (Interactive)")
    fig.show()

## 12. Missing Values

In [ ]:
# NULL counts
null_counts = df_v.isnull().sum()
null_pct = (df_v.isnull().sum() / len(df_v) * 100).round(1)
null_df = pd.DataFrame({"null_count": null_counts, "null_pct": null_pct})
null_df = null_df[null_df["null_count"] > 0].sort_values("null_pct", ascending=False)

print(f"Columns with missing values: {len(null_df)}")
print(null_df.to_string())

## 13. Key Findings Summary

### Target Distribution
- ~76.6% FULL_MATCH, ~23.4% ANY_VARIANCE → moderate imbalance (3.3:1)
- Binary classification is viable; multi-class is secondary experiment

### Strongest Signals (expected)
1. **Vendor quality_score** — directly drives match_rate in the generator (quality_factor)
2. **Vendor risk_score** — correlated with quality (r ≈ -0.575)
3. **Vendor historical match rate** — strongest proxy (but needs LOO to avoid leakage)
4. **Contract coverage** — on-contract POs have more stable pricing
5. **GR rejection indicators** — quality issues correlate with match failures

### Leakage Risks
- `price_variance`, `quantity_variance` — ARE the label
- `payment_block`, `block_reason`, `status` — derived from label
- `unit_price_invoiced`, `quantity_invoiced` — Mode A exclusion (pre-receipt)
- Vendor invoice behavior features need leave-one-out computation

### Modeling Recommendations
- Use `RepeatedStratifiedKFold(5, 3)` — temporal split not viable with 320 rows
- Binary-first approach, then multi-class as secondary
- Class weights rather than SMOTE (too few samples for synthetic oversampling)
- If F1 > 0.95, suspect leakage — the generator's match_rate formula has noise